In [ ]:
import tensorflow as tf
for g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(g, True)
    except: pass
print(tf.config.list_physical_devices('GPU'))


# StarDist 3D — Treino com cortes XY/ZX/ZY (ortoplanos)

Este notebook treina um modelo **StarDist 3D** para segmentar núcleos em volumes 3D (TIFF stacks), utilizando:

- **Pré-processamento**: normalização por percentis e *resampling* para voxels isotrópicos (recomendado se o Z-step for diferente de XY).
- **Aumento de dados**: geração de amostras em **três orientações** (XY, ZX e ZY) via permuta de eixos — isto expõe o modelo a cortes ortogonais sem perder a natureza 3D.
- **Treino/validação**: divisão automatizada, *checkpointing*, gráficos de *loss*.
- **Inferência e *benchmark***: predição num volume de teste, cálculo de métricas simples (Dice binário), e gravação de *masks* 3D.

**Requisitos**: GPU opcional mas recomendado. Pacotes: `stardist`, `csbdeep`, `tensorflow`, `tifffile`, `scikit-image`, `numpy`, `matplotlib`, `napari` (opcional para visualização).

**Dica**: Se vais anotar para o Cellpose, podes reutilizar **as mesmas máscaras** (cada núcleo com um rótulo inteiro único) para o StarDist 3D.


## 0) Instalação (executa uma vez no teu ambiente)

Executa a célula abaixo **no teu PC** (não aqui) para instalar dependências. Ajusta o TensorFlow GPU conforme a tua CUDA.


In [ ]:
# !pip install -U stardist csbdeep tensorflow tifffile scikit-image matplotlib napari[all]
# Em GPU: instala a versão TF compatível com a tua CUDA/cuDNN.
# Verifica CUDA: import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))

## 1) Configuração do projeto
Coloca os teus dados assim:
```
project/
 ├── images/
 │    ├── sample01.tif       # volume 3D (Z,Y,X)
 │    ├── sample02.tif
 │    └── ...
 └── masks/
      ├── sample01.tif       # mesma shape, labels int (cada núcleo = ID)
      ├── sample02.tif
      └── ...
```

Se as tuas anotações estiverem em **Cellpose .npy**, usa a célula de conversão (Secção 2.1) para gerar TIFF de labels.


In [ ]:
import os, glob, math, random, shutil
from pathlib import Path
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
from typing import Tuple, List

from skimage.transform import resize
from skimage.exposure import rescale_intensity

from stardist.models import Config3D, StarDist3D
from csbdeep.utils import normalize

# ============== PARÂMETROS PRINCIPAIS ==============
PROJECT_DIR = Path('project')        # <-- muda para a tua pasta
IMAGES_DIR  = PROJECT_DIR / 'images'
MASKS_DIR   = PROJECT_DIR / 'masks'
MODEL_BASE  = Path('models')         # onde o modelo será guardado
MODEL_NAME  = 'stardist3d_ortho'     # nome do modelo

# Tamanhos de voxel (em micrómetros, por exemplo). Ajusta para o teu microscópio.
VOXEL_SIZE_ZYX = (1.00, 0.30, 0.30)  # (Z, Y, X) original, e.g., Z=1.0 µm, XY=0.3 µm
TARGET_VOXEL_ZYX = (0.30, 0.30, 0.30)  # isotrópico recomendado
RESAMPLE_TO_ISOTROPIC = True          # se False, usa voxels originais (mas perde ortoplano consistente)

# Aumento por ortoplanos (permuta de eixos para gerar orientações XY, ZX, ZY)
ORTHOPLANE_AUG = True

# Split e treino
VALIDATION_FRACTION = 0.2
EPOCHS = 200
BATCH_SIZE = 2
PATCH_SIZE = (64, 128, 128)   # (Z, Y, X) em voxels após *resampling*
N_RAYS = 32                   # complexidade dos sólidos estrelados (maior = mais detalhado, mais pesado)
USE_TF32 = True               # em GPUs recentes (Ampere+), pode acelerar

random.seed(42)
np.random.seed(42)

print('Projetos em:', PROJECT_DIR.resolve())

## 2) Utilitários
Inclui normalização por percentis, *resampling* para voxel isotrópico e conversão de anotações Cellpose → TIFF labels.


In [ ]:
def read_volume(path):
    vol = tiff.imread(str(path))
    vol = np.asarray(vol)
    if vol.ndim != 3:
        raise ValueError(f'Esperado volume 3D (Z,Y,X), mas {path} tem shape {vol.shape}')
    return vol

def percentile_normalize(vol, pmin=1.0, pmax=99.8):
    vmin = np.percentile(vol, pmin)
    vmax = np.percentile(vol, pmax)
    if vmax <= vmin:
        vmax = vmin + 1e-6
    vol = np.clip((vol - vmin)/(vmax - vmin), 0, 1)
    return vol.astype(np.float32)

def resample_to_target_isotropic(vol, voxel_zyx, target_zyx, order=1):
    # calcula fatores de escala por eixo = voxel_original / voxel_target
    scale = np.array(voxel_zyx) / np.array(target_zyx)
    new_shape = np.round(np.array(vol.shape) * scale).astype(int)
    # resize espera ordem (Z,Y,X)
    out = resize(vol, new_shape, order=order, preserve_range=True, anti_aliasing=(order>0))
    return out.astype(vol.dtype)

def ensure_label_dtype(lbl):
    # garante int16/32 para labels, mantendo 0 como fundo
    if lbl.dtype.kind in 'iu':
        return lbl
    return lbl.astype(np.uint16)

def match_pairs(images_dir, masks_dir):
    imgs = sorted(glob.glob(str(images_dir / '*.tif'))) + sorted(glob.glob(str(images_dir / '*.tiff')))
    X_paths, Y_paths = [], []
    for ip in imgs:
        name = Path(ip).stem
        cand = [m for m in [masks_dir / f'{name}.tif', masks_dir / f'{name}.tiff'] if m.exists()]
        if not cand:
            print(f'[AVISO] Sem máscara correspondente para {ip}')
            continue
        X_paths.append(ip)
        Y_paths.append(str(cand[0]))
    return X_paths, Y_paths

def add_orthoplane_views(vol, lbl):
    # retorna lista [(vol, lbl)] com 1 (ZYX) + 2 permutas (YZX, XZY) reordenadas para ZYX
    out_v, out_l = [vol], [lbl]
    # ZX (permuta Z<->Y): eixos (Y,Z,X) -> reordenar para (Z,Y,X)
    v_yzx = np.transpose(vol, (1,0,2))
    l_yzx = np.transpose(lbl, (1,0,2))
    # reordenado já é (Z,Y,X) porque o primeiro eixo passou a ser Y
    out_v.append(v_yzx)
    out_l.append(l_yzx)
    # ZY (permuta Z<->X): eixos (X,Y,Z) -> reordenar para (Z,Y,X)
    v_xyz = np.transpose(vol, (2,1,0))
    l_xyz = np.transpose(lbl, (2,1,0))
    out_v.append(v_xyz)
    out_l.append(l_xyz)
    return out_v, out_l

def train_val_split(n, val_frac=0.2, seed=42):
    idx = np.arange(n)
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    n_val = max(1, int(round(n*val_frac)))
    val_idx = idx[:n_val]
    tr_idx = idx[n_val:]
    return tr_idx, val_idx

def save_tif(path, vol, dtype=None):
    vol = vol if dtype is None else vol.astype(dtype)
    tiff.imwrite(str(path), vol, imagej=True)

def dice_bin(pred, gt):
    # Dice simples binário (não instancia). Útil como sanity check.
    p = pred>0
    g = gt>0
    inter = np.logical_and(p,g).sum()
    den = p.sum()+g.sum()
    return (2*inter/den) if den>0 else 1.0


### 2.1) Converter anotações Cellpose (*.npy) para TIFF de labels (opcional)
Se tens máscaras do Cellpose em `*.npy`, usa esta célula para gerar TIFFs de labels compatíveis.


In [ ]:
def convert_cellpose_npy_folder(npy_dir, out_dir, pattern='*.npy', key='masks'):
    npy_dir = Path(npy_dir); out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    files = sorted(npy_dir.glob(pattern))
    print(f'Encontrados {len(files)} ficheiros .npy')
    for f in files:
        data = np.load(f, allow_pickle=True)
        if isinstance(data, np.lib.npyio.NpzFile):
            if key not in data: raise KeyError(f'chave {key} não encontrada em {f}')
            masks = data[key]
        else:
            # alguns salvam como dict
            if isinstance(data.item(), dict):
                masks = data.item().get(key, None)
                if masks is None: raise KeyError(f'chave {key} não encontrada em {f}')
            else:
                # pode ser diretamente a matriz de masks
                masks = data
        masks = ensure_label_dtype(masks)
        out_path = out_dir / (f.stem + '.tif')
        save_tif(out_path, masks, dtype=np.uint16)
    print('Conversão concluída.')

# Exemplo de uso:
# convert_cellpose_npy_folder('cellpose_npy_masks', MASKS_DIR)

## 3) Carregar dados, normalizar e (opcional) *resampling* para voxels isotrópicos
Se `RESAMPLE_TO_ISOTROPIC=True`, cada volume é reamostrado para `TARGET_VOXEL_ZYX`. As máscaras usam interpolação `order=0` (vizinho mais próximo) para preservar rótulos inteiros.


In [ ]:
X_paths, Y_paths = match_pairs(IMAGES_DIR, MASKS_DIR)
assert len(X_paths)==len(Y_paths) and len(X_paths)>0, 'Sem pares imagem/máscara!'
print(f'{len(X_paths)} pares encontrados.')

X_vols, Y_vols = [], []
for xp, yp in zip(X_paths, Y_paths):
    x = read_volume(xp)
    y = read_volume(yp)
    if x.shape != y.shape:
        raise ValueError(f'Shapes diferentes: {xp} {x.shape} vs {yp} {y.shape}')
    # normalização
    x = percentile_normalize(x)
    # resampling para isotrópico se desejado
    if RESAMPLE_TO_ISOTROPIC:
        x = resample_to_target_isotropic(x, VOXEL_SIZE_ZYX, TARGET_VOXEL_ZYX, order=1)
        y = resample_to_target_isotropic(y, VOXEL_SIZE_ZYX, TARGET_VOXEL_ZYX, order=0)
    X_vols.append(x.astype(np.float32))
    Y_vols.append(ensure_label_dtype(y))

print('Exemplo shape pós-processamento:', X_vols[0].shape, Y_vols[0].shape)

### 3.1) Aumento por ortoplanos (opcional)
Gera versões reorientadas (permuta de eixos) para expor o modelo a cortes **XY, ZX, ZY** mantendo o treino 3D.


In [ ]:
X_aug, Y_aug = [], []
for x, y in zip(X_vols, Y_vols):
    if ORTHOPLANE_AUG:
        vs, ls = add_orthoplane_views(x, y)
        X_aug.extend(vs)
        Y_aug.extend(ls)
    else:
        X_aug.append(x)
        Y_aug.append(y)

print('Total de volumes para treino (incluindo ortoplanos):', len(X_aug))

## 4) *Split* treino/validação


In [ ]:
tr_idx, val_idx = train_val_split(len(X_aug), val_frac=VALIDATION_FRACTION, seed=42)
X_tr = [X_aug[i] for i in tr_idx]
Y_tr = [Y_aug[i] for i in tr_idx]
X_val = [X_aug[i] for i in val_idx]
Y_val = [Y_aug[i] for i in val_idx]
print(f'Treino: {len(X_tr)} | Validação: {len(X_val)}')

## 5) Configurar e treinar o StarDist 3D
Com voxels isotrópicos, definimos `anisotropy=(1,1,1)`. Se **não** reamostrares para isotrópico, podes definir `anisotropy=(z/xy,1,1)` — porém isso fica inconsistente com a permuta de eixos dos ortoplanos num único modelo. Por isso **recomenda-se isotropizar** antes.


In [ ]:
if USE_TF32:
    try:
        import tensorflow as tf
        tf.keras.mixed_precision.set_global_policy('float32')
        # Em GPUs Ampere+, TF32 costuma estar ativo por padrão; deixamos como está.
    except Exception as e:
        print('[AVISO] TensorFlow não disponível para configurar TF32:', e)

anisotropy = (1,1,1) if RESAMPLE_TO_ISOTROPIC else (VOXEL_SIZE_ZYX[0]/VOXEL_SIZE_ZYX[1], 1, 1)
conf = Config3D(
    rays=N_RAYS,
    anisotropy=anisotropy,
    grid=(1,1,1),
    n_channel_in=1,
    train_patch_size=PATCH_SIZE,
    train_batch_size=BATCH_SIZE,
)
print(conf)

model = StarDist3D(conf, name=MODEL_NAME, basedir=str(MODEL_BASE))

history = model.train(
    X_tr, Y_tr,
    validation_data=(X_val, Y_val),
    epochs=EPOCHS,
)

# guarda gráfico de loss
plt.figure()
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.grid(True)
os.makedirs(MODEL_BASE/Path(MODEL_NAME), exist_ok=True)
plt.savefig(MODEL_BASE/Path(MODEL_NAME)/'training_curve.png', dpi=150)
plt.show()

### 5.1) Exportar modelo (SavedModel) e info
Exporta o modelo em formato TensorFlow (útil para uso em `stardist-napari` ou *serving*).


In [ ]:
try:
    model.export_TF()
    print('Modelo exportado para TensorFlow SavedModel.')
except Exception as e:
    print('Falha ao exportar SavedModel (opcional):', e)

## 6) Inferência num volume de teste + métrica Dice (binária)
Como *sanity check*, calculamos o Dice binário (não por instância). Para métricas de instância (AJI, PQ, etc.), integra com o teu pipeline de *benchmark*.


In [ ]:
test_vol = X_val[0]
test_gt  = Y_val[0]

labels, details = model.predict_instances(test_vol)
d = dice_bin(labels, test_gt)
print('Dice binário (sanity check):', round(d,4))

# salvar predição
pred_path = PROJECT_DIR / 'pred_example.tif'
save_tif(pred_path, labels.astype(np.uint16))
print('Predição salva em:', pred_path)

### 6.1) Visualização no Napari (opcional)
Corre localmente (não em ambientes headless) para sobrepor `Raw` e `Labels`.


In [ ]:
# import napari
# v = napari.Viewer()
# v.add_image(test_vol, name='Raw', blending='additive')
# v.add_labels(labels, name='StarDist3D')
# napari.run()

## 7) *Benchmark* rápido em toda a validação
Calcula Dice binário por volume e grava um CSV.


In [ ]:
import csv
rows = [('idx','dice_bin')]
for i,(xv,yv) in enumerate(zip(X_val, Y_val)):
    pred, _ = model.predict_instances(xv)
    rows.append((i, float(dice_bin(pred, yv))))
csv_path = PROJECT_DIR / 'val_dice_binary.csv'
with open(csv_path, 'w', newline='') as f:
    cw = csv.writer(f); cw.writerows(rows)
print('Métricas salvas em:', csv_path)